# SinoNom OCR — Phase 2: Spatial Layout Analysis

**HCMUS NaturalLanguageProcessing — Midterm Project**  
**An Nam Nhất Thống Chí (HVH_004)**

---

This notebook handles **Phase 2** of the pipeline: running the Adaptive Horizontal Threshold (AHT) layout clustering algorithm to organize raw text bounding boxes (derived from OCR) into right-to-left vertical text columns.


## 0 · Environment Setup


In [ ]:
# ─── Detect environment ────────────────────────────────────────────────────
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python {sys.version}")

In [ ]:
# ─── Google Drive Mount & Project Path Discovery (Colab only) ────────────────
if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    repo_name = "SinoNomViet_Transliteration_OCR"
    repo_url = f"https://github.com/khang3004/{repo_name}.git"
    candidate_files = [
        "src/sinonom_ocr/data_scraper.py",
        "src/sinonom_ocr/spatial_layout_engine.py",
        "src/sinonom_ocr/alignment_validator.py",
    ]

    def has_project_files(path: Path) -> bool:
        return all((path / f).exists() for f in candidate_files)

    found_root = None

    # 1. Check if files are already in current directory or a subfolder of /content
    cwd = Path(os.getcwd())
    if has_project_files(cwd):
        found_root = cwd
    else:
        for p in Path("/content").glob("**/src/sinonom_ocr/data_scraper.py"):
            if has_project_files(p.parent.parent.parent):
                found_root = p.parent.parent.parent
                break

    # 2. If not found locally, try to find them on Google Drive
    if not found_root:
        drive_mounted = os.path.exists("/content/drive/MyDrive")
        if not drive_mounted:
            print("❓ Google Drive is not mounted. Mounting now...")
            try:
                from google.colab import drive

                drive.mount("/content/drive")
                drive_mounted = True
            except Exception as e:
                print(f"⚠️ Google Drive mount failed or skipped: {e}")

        if drive_mounted:
            drive_paths = [
                Path("/content/drive/MyDrive/SinoNomOCR/HVH_004"),
                Path(f"/content/drive/MyDrive/{repo_name}"),
                Path("/content/drive/MyDrive/SinoNomVietnamese_OCR_Project"),
                Path("/content/drive/MyDrive/SinoNomViet_Transliteration_OCR"),
            ]
            for dp in drive_paths:
                if dp.exists() and has_project_files(dp):
                    found_root = dp
                    print(f"🎯 Found project files in Drive: {found_root}")
                    break

            if not found_root:
                print("🔍 Searching Google Drive for project files...")
                for p in Path("/content/drive/MyDrive").glob("**/src/sinonom_ocr/data_scraper.py"):
                    if has_project_files(p.parent.parent.parent):
                        found_root = p.parent.parent.parent
                        print(f"🎯 Found project files in Drive subfolder: {found_root}")
                        break

    # 3. If still not found, clone the repository from GitHub
    if not found_root:
        print("📁 Project files not found. Cloning repository from GitHub...")
        try:
            subprocess.run(["git", "clone", repo_url, f"/content/{repo_name}"], check=True)
            found_root = Path(f"/content/{repo_name}")
            print("✅ Repository cloned successfully.")
        except Exception as e:
            print(f"❌ Failed to clone repository: {e}")

    if found_root:
        PROJECT_ROOT = str(found_root)
        os.chdir(PROJECT_ROOT)
        print(f"🎯 Project root set to: {PROJECT_ROOT}")
        print(f"🔄 Changed working directory to: {os.getcwd()}")
    else:
        PROJECT_ROOT = "/content"
        print("❌ ERROR: Could not locate project root.")
else:
    # Local Jupyter running in notebooks/ directory
    PROJECT_ROOT = os.path.abspath("..")
    os.chdir(PROJECT_ROOT)
    print(f"🎯 Local project root: {PROJECT_ROOT}")
    print(f"🔄 Changed working directory to: {os.getcwd()}")

In [ ]:
# ─── Install package & dependencies ────────────────────────────────────────
# Installs the package in editable mode, which resolves all dependencies from pyproject.toml
if IN_COLAB:
    # Install the package in editable mode to register sinonom_ocr package
    %pip install -q -e .
    print("✅ Package and dependencies installed on Colab.")
else:
    print("✅ Local environment is managed by uv (run 'make install' if needed).")

In [ ]:
# ─── Path configuration ────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path(PROJECT_ROOT)

# Add src/ directory to sys.path so we can import our sinonom_ocr package
SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Output directories
RAW_IMAGES_DIR = ROOT / "data" / "raw_images"
OUTPUT_XML_DIR = ROOT / "output" / "xml"
OUTPUT_XLSX_DIR = ROOT / "output" / "excel"
DICT_DIR = ROOT / "data" / "dicts"

for d in [RAW_IMAGES_DIR, OUTPUT_XML_DIR, OUTPUT_XLSX_DIR, DICT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directory structure initialized.")

---

## 1 · Import Layout Module


In [ ]:
import logging

# Pipeline modules
from sinonom_ocr.spatial_layout_engine import (
    BoundingBox,
    create_mock_multi_char_response,
    create_mock_ocr_response,
    process_page_layout,
)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(name)s | %(message)s",
)

print("✅ Spatial Layout Engine module imported successfully.")

---

## 4 · Mock OCR + Spatial Layout Processing

In production, replace `create_mock_pages()` with real OCR API calls.


In [ ]:
# ─── Mock corpus definition ────────────────────────────────────────────────
# Represents a mini-corpus with 2 chapters, each with 2 pages.
# Each page contains a list of BoundingBox objects with OCR text.
# Structure: corpus[chapter_idx][page_idx] = list[BoundingBox]


def build_mock_corpus() -> list[list[list[BoundingBox]]]:
    """Build a mock corpus with realistic SinoNom content.

    Returns:
        3-D list: [chapter][page][box]
    """
    # Chapter 1, Page 1 — 5-column single-character example (Prof. Dien's PDF)
    ch1_pg1 = create_mock_ocr_response()  # 百 年 身 後 名

    # Chapter 1, Page 2 — multi-character columns
    ch1_pg2 = create_mock_multi_char_response()  # 帝王臨朝 / 聖德日 / 新月異盛

    # Chapter 2, Page 1 — another sample page
    ch2_pg1 = [
        BoundingBox.from_xyxy(320, 10, 360, 70, text="安", confidence=0.95),
        BoundingBox.from_xyxy(320, 75, 360, 135, text="南", confidence=0.93),
        BoundingBox.from_xyxy(240, 10, 280, 70, text="一", confidence=0.97),
        BoundingBox.from_xyxy(240, 75, 280, 135, text="統", confidence=0.91),
        BoundingBox.from_xyxy(160, 10, 200, 70, text="志", confidence=0.94),
        BoundingBox.from_xyxy(160, 75, 200, 135, text="卷", confidence=0.88),
    ]

    # Chapter 2, Page 2 — historical records style
    ch2_pg2 = [
        BoundingBox.from_xyxy(310, 10, 350, 60, text="皇", confidence=0.96),
        BoundingBox.from_xyxy(310, 65, 350, 115, text="帝", confidence=0.94),
        BoundingBox.from_xyxy(230, 10, 270, 60, text="詔", confidence=0.92),
        BoundingBox.from_xyxy(230, 65, 270, 115, text="曰", confidence=0.90),
        BoundingBox.from_xyxy(150, 10, 190, 60, text="天", confidence=0.95),
        BoundingBox.from_xyxy(150, 65, 190, 115, text="下", confidence=0.93),
    ]

    return [
        [ch1_pg1, ch1_pg2],  # Chapter 1
        [ch2_pg1, ch2_pg2],  # Chapter 2
    ]


MOCK_CORPUS = build_mock_corpus()

print("Mock corpus structure:")
for ch_idx, chapter in enumerate(MOCK_CORPUS, 1):
    for pg_idx, page_boxes in enumerate(chapter, 1):
        print(f"  Chapter {ch_idx}, Page {pg_idx}: {len(page_boxes)} bboxes")

In [ ]:
# ─── Run spatial layout engine on all pages ────────────────────────────────

# corpus_layout[chapter_idx][page_idx] = (columns, ordered_boxes)
corpus_layout: list[list[tuple]] = []

for ch_idx, chapter in enumerate(MOCK_CORPUS, 1):
    ch_layout = []
    for pg_idx, raw_boxes in enumerate(chapter, 1):
        columns, ordered_boxes = process_page_layout(
            raw_boxes,
            alpha=0.5,
            min_boxes_per_column=1,
        )
        ch_layout.append((columns, ordered_boxes))
        print(
            f"Ch{ch_idx} Pg{pg_idx}: "
            f"{len(raw_boxes)} boxes → "
            f"{len(columns)} columns  |  "
            f"Reading order: {' → '.join(b.text for b in ordered_boxes)}"
        )
    corpus_layout.append(ch_layout)

print("\n✅ Spatial layout processing complete.")

---

## 5 · Export Layout Output for Phase 3

We serialize the resulting column structures and bounding boxes to `data/ocr_layout_output.json` so that they can be processed by Phase 3 (Text Alignment & Validation) without needing to re-run the OCR or layout detection step.


In [ ]:
# ─── Serialize layout to JSON ──────────────────────────────────────────────
import json

output_file = ROOT / "data" / "ocr_layout_output.json"

serialized_corpus = []
for ch_idx, ch_layout in enumerate(corpus_layout, start=1):
    ch_data = []
    for pg_idx, (columns, ordered_boxes) in enumerate(ch_layout, start=1):
        pg_data = {
            "chapter": ch_idx,
            "page": pg_idx,
            "columns": [
                {
                    "column_id": col.column_id,
                    "x_center": col.x_center,
                    "x_span": col.x_span,
                    "boxes": [
                        {
                            "raw_polygon": box.raw_polygon,
                            "text": box.text,
                            "confidence": box.confidence,
                            "column_id": box.column_id,
                            "reading_idx": box.reading_idx,
                        }
                        for box in col.boxes
                    ],
                }
                for col in columns
            ],
            "ordered_boxes": [
                {
                    "raw_polygon": box.raw_polygon,
                    "text": box.text,
                    "confidence": box.confidence,
                    "column_id": box.column_id,
                    "reading_idx": box.reading_idx,
                }
                for box in ordered_boxes
            ],
        }
        ch_data.append(pg_data)
    serialized_corpus.append(ch_data)

with open(output_file, "w", encoding="utf-8") as fh:
    json.dump(serialized_corpus, fh, ensure_ascii=False, indent=2)

print(
    f"✅ Successfully exported spatial layout results of {len(serialized_corpus)} chapters to {output_file}"
)